In [1]:
# Apollo Hospital Appointment No-Show and Patient Engagement Analysis

## Python Exploratory Data Analysis (EDA)

##Objective:** Analyze 75,000 hospital appointments to identify no-show patterns, patient engagement trends, financial impact, and doctor utilization between 2022 and 2024.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import warnings

warnings.filterwarnings("ignore")

In [3]:
# Load the datasets

appointments = pd.read_csv("apollo_appointments_fact.csv")
doctors = pd.read_csv("apollo_doctors_dim.csv")

In [4]:
appointments.head()

,appointment_id,patient_id,doctor_id,appointment_date,appointment_day,appointment_day_of_week,appointment_month,appointment_quarter,appointment_year,appointment_hour,...,insurance_coverage,insurance_provider,insurance_covered_amount,patient_out_of_pocket,revenue_realized,payment_mode,wait_time_minutes,consultation_duration_min,patient_satisfaction_score,doctor_utilization_pct
0,APPT5000001,PAT117608,DOC1035,2023-11-15,Wednesday,2,11,Q4,2023,15,...,No,NaN,0,2700,2700,UPI,11.0,40.0,4.8,84.2
1,APPT5000002,PAT128732,DOC1049,2023-03-13,Monday,0,3,Q1,2023,9,...,No,NaN,0,1400,1400,UPI,21.0,18.0,4.1,64.5
2,APPT5000003,PAT137532,DOC1033,2022-02-07,Monday,0,2,Q1,2022,17,...,No,NaN,0,600,600,Net Banking,6.0,6.0,5.0,75.6
3,APPT5000004,PAT113284,DOC1288,2023-11-20,Monday,0,11,Q4,2023,17,...,No,NaN,0,1275,1275,UPI,2.0,25.0,4.4,68.6
4,APPT5000005,PAT107443,DOC1064,2024-02-02,Friday,4,2,Q1,2024,19,...,No,NaN,0,1700,1700,Cash,7.0,13.0,3.5,67.7


In [7]:
doctors.head()


,doctor_id,doctor_name,specialty,qualification,experience_years,city,state,hospital_name,consultation_fee,rating,total_reviews,available_days,avg_slot_duration_min,accepts_insurance,teleconsult_enabled
0,DOC1001,Dr. Deepa Sharma,Urology,"MBBS, MCh",12,Bengaluru,Karnataka,Apollo Hospital Bannerghatta,2300,4.4,1502,Mon-Fri,15,Yes,No
1,DOC1002,Dr. Venkatesan Khan,Paediatrics,"MBBS, MS, FRCS",25,Chandigarh,Punjab,Apollo Clinic Sector 8,1200,4.5,2453,Mon-Sat,10,Yes,Yes
2,DOC1003,Dr. Pradeep Menon,General Physician,"MBBS, DM",13,Mumbai,Maharashtra,Apollo Clinic Andheri,700,4.7,995,Mon-Wed-Fri-Sat,15,No,No
3,DOC1004,Dr. Vivek Mathew,General Physician,"MBBS, MD",31,Mumbai,Maharashtra,Apollo Hospital Navi Mumbai,1000,4.9,666,Mon-Wed-Fri-Sat,20,Yes,Yes
4,DOC1005,Dr. Savita Philip,General Physician,"MBBS, MD, PhD",5,Mumbai,Maharashtra,Apollo Hospital Navi Mumbai,600,4.5,1383,Mon-Sat,30,Yes,Yes


In [ ]:
## the size of the DataFrame.
print("Appointments Dataset:", appointments.shape)
print("Doctors Dataset:", doctors.shape)

Appointments Dataset: (75000, 52)
Doctors Dataset: (320, 15)


In [12]:
## Dataset Structure
##Understanding the available columns before starting data cleaning.

In [13]:
# Display all column names

print("Appointments Columns:")
print(appointments.columns.tolist())

print("\nDoctors Columns:")
print(doctors.columns.tolist())

Appointments Columns:
['appointment_id', 'patient_id', 'doctor_id', 'appointment_date', 'appointment_day', 'appointment_day_of_week', 'appointment_month', 'appointment_quarter', 'appointment_year', 'appointment_hour', 'appointment_minute', 'time_slot', 'time_of_day', 'booking_date', 'booking_lead_days', 'booking_channel', 'appointment_type', 'specialty', 'city', 'state', 'hospital_name', 'patient_age', 'age_group', 'patient_gender', 'patient_city_tier', 'patient_chronic_condition', 'apollo_member', 'membership_type', 'reminder_type', 'reminders_sent', 'patient_prior_visits', 'patient_prior_no_shows', 'is_first_visit', 'repeat_visit', 'visit_reason', 'appointment_status', 'no_show', 'no_show_flag', 'cancellation_reason', 'consultation_fee', 'discount_pct', 'actual_fee_charged', 'insurance_coverage', 'insurance_provider', 'insurance_covered_amount', 'patient_out_of_pocket', 'revenue_realized', 'payment_mode', 'wait_time_minutes', 'consultation_duration_min', 'patient_satisfaction_score',

In [15]:
## Data Types
##Checking the data type and non-null count of every column.

In [16]:
appointments.info()

<class 'pandas.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 52 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   appointment_id              75000 non-null  str    
 1   patient_id                  75000 non-null  str    
 2   doctor_id                   75000 non-null  str    
 3   appointment_date            75000 non-null  str    
 4   appointment_day             75000 non-null  str    
 5   appointment_day_of_week     75000 non-null  int64  
 6   appointment_month           75000 non-null  int64  
 7   appointment_quarter         75000 non-null  str    
 8   appointment_year            75000 non-null  int64  
 9   appointment_hour            75000 non-null  int64  
 10  appointment_minute          75000 non-null  int64  
 11  time_slot                   75000 non-null  str    
 12  time_of_day                 75000 non-null  str    
 13  booking_date                75000 non-null

In [18]:
doctors.info()

<class 'pandas.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   doctor_id              320 non-null    str    
 1   doctor_name            320 non-null    str    
 2   specialty              320 non-null    str    
 3   qualification          320 non-null    str    
 4   experience_years       320 non-null    int64  
 5   city                   320 non-null    str    
 6   state                  320 non-null    str    
 7   hospital_name          320 non-null    str    
 8   consultation_fee       320 non-null    int64  
 9   rating                 320 non-null    float64
 10  total_reviews          320 non-null    int64  
 11  available_days         320 non-null    str    
 12  avg_slot_duration_min  320 non-null    int64  
 13  accepts_insurance      320 non-null    str    
 14  teleconsult_enabled    320 non-null    str    
dtypes: float64(1), in

In [19]:
## Missing Values
##Identifying which columns contain null values.

In [20]:
missing = appointments.isnull().sum()

missing[missing > 0]

patient_chronic_condition     28038
membership_type               58225
no_show                        2348
cancellation_reason           69207
insurance_provider            51512
wait_time_minutes             19725
consultation_duration_min     19725
patient_satisfaction_score    19725
doctor_utilization_pct        19725
dtype: int64

In [21]:
## Duplicate Records
##Checking whether duplicate rows exist.

In [22]:
print("Appointment Duplicates:", appointments.duplicated().sum())
print("Doctor Duplicates:", doctors.duplicated().sum())

Appointment Duplicates: 0
Doctor Duplicates: 0


In [23]:
## Data Cleaning

##Converting date columns into proper datetime format for time-based analysis.

In [24]:
# Convert text dates into datetime format

appointments["booking_date"] = pd.to_datetime(appointments["booking_date"])
appointments["appointment_date"] = pd.to_datetime(appointments["appointment_date"])

In [25]:
## checking the changes
appointments[["booking_date", "appointment_date"]].info()

<class 'pandas.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   booking_date      75000 non-null  datetime64[us]
 1   appointment_date  75000 non-null  datetime64[us]
dtypes: datetime64[us](2)
memory usage: 1.1 MB


In [26]:
## Merge Fact and Dimension Tables

##Joining doctor information using `doctor_id`.

df = appointments.merge(doctors, on="doctor_id", how="left")

In [27]:
## checking merge dataset
print("Merged Dataset:", df.shape)
df.head()

Merged Dataset: (75000, 66)


,appointment_id,patient_id,doctor_id,appointment_date,appointment_day,appointment_day_of_week,appointment_month,appointment_quarter,appointment_year,appointment_hour,...,city_y,state_y,hospital_name_y,consultation_fee_y,rating,total_reviews,available_days,avg_slot_duration_min,accepts_insurance,teleconsult_enabled
0,APPT5000001,PAT117608,DOC1035,2023-11-15,Wednesday,2,11,Q4,2023,15,...,Bengaluru,Karnataka,Apollo Clinic Jayanagar,2700,4.6,55,Mon-Fri,30,Yes,No
1,APPT5000002,PAT128732,DOC1049,2023-03-13,Monday,0,3,Q1,2023,9,...,Pune,Maharashtra,Apollo Spectra Pune,1400,4.7,1912,Mon-Fri,15,No,Yes
2,APPT5000003,PAT137532,DOC1033,2022-02-07,Monday,0,2,Q1,2022,17,...,Bengaluru,Karnataka,Apollo Spectra Koramangala,600,4.7,589,Mon-Fri,10,Yes,No
3,APPT5000004,PAT113284,DOC1288,2023-11-20,Monday,0,11,Q4,2023,17,...,Delhi,Delhi,Apollo Hospital Sarita Vihar,1500,4.9,2281,Mon-Fri,20,Yes,Yes
4,APPT5000005,PAT107443,DOC1064,2024-02-02,Friday,4,2,Q1,2024,19,...,Coimbatore,Tamil Nadu,Apollo Clinic Coimbatore,2000,4.9,393,Mon-Wed-Fri-Sat,20,No,No


In [29]:
## Business Overview

### 1. Appointment Volume Trend (2022–2024)

###Analyzing how appointment bookings changed across months and years.

In [30]:
# Count appointments for each month in each year

monthly_trend = (
    df.groupby(["appointment_year", "appointment_month"])
      .size()
      .reset_index(name="total_appointments")
)

monthly_trend.head()

,appointment_year,appointment_month,total_appointments
0,2022,1,2426
1,2022,2,2012
2,2022,3,2085
3,2022,4,1926
4,2022,5,2033


In [31]:
month_names = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sep", 10: "Oct", 11: "Nov", 12: "Dec"
}

monthly_trend["Month_Name"] = monthly_trend["appointment_month"].map(month_names)

monthly_trend.head()

,appointment_year,appointment_month,total_appointments,Month_Name
0,2022,1,2426,Jan
1,2022,2,2012,Feb
2,2022,3,2085,Mar
3,2022,4,1926,Apr
4,2022,5,2033,May


In [34]:
## monthly trend 

fig = px.line(
    monthly_trend,
    x="Month_Name",
    y="total_appointments",
    color="appointment_year",
    markers=True,
    title="Monthly Appointment Trend (2022–2024)"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

In [ ]:
### 2. Quarterly Appointment Volume Trend (2022–2024)

## Analyzing how appointment volume changes across quarters from 2022 to 2024.

quarterly_trend = (
    df.groupby(["appointment_year", "appointment_quarter"])
      .size()
      .reset_index(name="total_appointments")
)

fig = px.bar(
    quarterly_trend,
    x="appointment_quarter",
    y="total_appointments",
    color="appointment_year",
    barmode="group",
    title="Quarterly Appointment Trend (2022–2024)"
)

fig.update_layout(template="plotly_white", title_x=0.5)

fig.show()

In [43]:
### Business Insight

#- Appointment volume trends can be compared across 2022, 2023, and 2024.
#- Peak months indicate periods of higher patient demand.
#- Low-volume months may require fewer resources, while peak months may require additional doctor availability and staff planning.
#- Compare appointment volume across Q1, Q2, Q3, and Q4.
#- Identify which quarter consistently receives higher patient demand.
#- Quarterly trends help Apollo plan staffing and doctor schedules more effectively.

In [44]:
### 2. Appointment Status Distribution

##Analyzing the distribution of appointment outcomes across Completed, No-Show, Cancelled, and Scheduled appointments.

In [45]:
# Count appointments by status

status_distribution = (
    df["appointment_status"]
      .value_counts()
      .reset_index()
)

# Rename columns for better readability

status_distribution.columns = [
    "appointment_status",
    "total_appointments"
]

status_distribution

,appointment_status,total_appointments
0,Completed,55275
1,No-Show,11584
2,Cancelled,5793
3,Scheduled,2348


In [46]:
fig = px.pie(
    status_distribution,
    names="appointment_status",
    values="total_appointments",
    title="Appointment Outcome Distribution",
    hole=0.4
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

In [47]:
### Business Insight

#- The chart shows the distribution of appointments across Completed, No-Show, Cancelled, and Scheduled statuses.
#- Completed appointments represent successful consultations, while No-Show and Cancelled appointments indicate operational losses.
#- Understanding this distribution helps Apollo identify the scale of missed appointments before investigating their causes in the next section.

In [ ]:
### 3. Booking Channel Distribution

##Which booking channels drive the highest volume and how does the channel mix look? 
 

In [49]:
# Count appointments by booking channel

channel_distribution = (
    df.groupby("booking_channel")
      .size()
      .reset_index(name="total_appointments")
      .sort_values(by="total_appointments", ascending=False)
)

channel_distribution

,booking_channel,total_appointments
0,Apollo App,34042
4,Website,18545
1,Call Centre,11201
3,Walk-In,7501
2,Partner App,3711


In [50]:
fig = px.bar(
    channel_distribution,
    x="booking_channel",
    y="total_appointments",
    color="booking_channel",
    text="total_appointments",
    title="Appointment Volume by Booking Channel"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    showlegend=False,
    xaxis_title="Booking Channel",
    yaxis_title="Total Appointments"
)

fig.update_traces(textposition="outside")

fig.show()

In [51]:
### Business Insight

#- The chart compares appointment volume across all booking channels.
#- The highest-volume channel represents the primary source of patient bookings.
#- Understanding the channel mix helps Apollo prioritize marketing efforts and improve booking experiences on the most-used platforms.

In [52]:
# No-Show Analysis

##This section identifies where and why patients miss appointments by analyzing no-show patterns across specialties, cities, booking behavior, appointment timing, and booking channels.

In [53]:
### 1. No-Show Rate by Specialty and City

##Comparing no-show rates across medical specialties and cities while excluding scheduled appointments.

In [54]:
# Exclude scheduled appointments before calculating no-show rate

no_show_df = df[
    df["appointment_status"] != "Scheduled"
].copy()

no_show_df.shape

(72652, 66)

In [64]:
# Count total appointments for each specialty
total = no_show_df.groupby("specialty_x").size()

# Count only No Show appointments for each specialty
no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]
    .groupby("specialty_x")
    .size()
)

# Combine both counts into one table
specialty_noshow = pd.DataFrame({
    "total_appointments": total,
    "no_show_count": no_show
}).fillna(0)

# Convert no_show_count to integer
specialty_noshow["no_show_count"] = specialty_noshow["no_show_count"].astype(int)

# Calculate No Show Rate
specialty_noshow["no_show_rate"] = (
    specialty_noshow["no_show_count"]
    / specialty_noshow["total_appointments"]
) * 100

# Convert specialty from index to column
specialty_noshow = specialty_noshow.reset_index()

# Sort from highest to lowest No Show Rate
specialty_noshow = specialty_noshow.sort_values(
    by="no_show_rate",
    ascending=False
)

# Show top 5 specialties
specialty_noshow.head()

,specialty_x,total_appointments,no_show_count,no_show_rate
11,Psychiatry,2900,711,24.517241
1,Dermatology,7566,1608,21.252974
2,ENT,5211,988,18.959893
3,Endocrinology,2024,355,17.539526
8,Ophthalmology,3103,503,16.210119


In [65]:
fig = px.bar(
    specialty_noshow,
    x="no_show_rate",
    y="specialty_x",
    orientation="h",
    text="no_show_rate",
    color="no_show_rate",
    title="No-Show Rate by Specialty"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="No-Show Rate (%)",
    yaxis_title="specialty_x"
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [66]:
### Business Insight

#- The chart ranks specialties by their no-show rate.
#- Specialties with higher no-show rates may require stronger reminder strategies.
#- Lower no-show specialties indicate better patient attendance.

In [67]:
### No-Show Rate by City

#Analyzing which cities have the highest and lowest no-show rates.

In [68]:
# Total appointments in each city
city_total = no_show_df["city_x"].value_counts()

# Only no-show appointments
city_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["city_x"]
    .value_counts()
)

# Combine both counts
city_noshow = pd.concat([city_total, city_no_show], axis=1)

# Rename columns
city_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values with 0
city_noshow = city_noshow.fillna(0)

# Convert no_show_count to integer
city_noshow["no_show_count"] = city_noshow["no_show_count"].astype(int)

# Calculate no-show rate
city_noshow["no_show_rate"] = (
    city_noshow["no_show_count"] /
    city_noshow["total_appointments"]
) * 100

# Convert index into column
city_noshow = city_noshow.reset_index()

# Rename city column
city_noshow = city_noshow.rename(columns={"index": "city"})

# Sort from highest to lowest
city_noshow = city_noshow.sort_values(
    "no_show_rate",
    ascending=False
)

city_noshow.head()

,city_x,total_appointments,no_show_count,no_show_rate
8,Lucknow,3001,516,17.194269
1,Mumbai,9734,1659,17.043353
0,Delhi,10404,1754,16.858900
14,Bhopal,2001,336,16.791604
5,Kolkata,5701,933,16.365550


In [70]:
fig = px.bar(
    city_noshow,
    x="no_show_rate",
    y="city_x",
    orientation="h",
    text="no_show_rate",
    color="no_show_rate",
    title="No-Show Rate by City"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="No-Show Rate (%)",
    yaxis_title="City"
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [71]:
### Business Insight

#- The chart ranks cities by their no-show rate.
#- Cities with higher no-show rates may require localized reminder campaigns and scheduling improvements.
#- Lower no-show cities indicate stronger patient attendance and engagement.

In [72]:
### 2. Booking Lead Time vs No-Show Rate

#Analyzing whether patients who book appointments further in advance are more likely to miss their appointments.

In [73]:
# Create booking lead time groups

bins = [-1, 1, 3, 7, 14, 30, 1000]

labels = [
    "0-1 Days",
    "2-3 Days",
    "4-7 Days",
    "8-14 Days",
    "15-30 Days",
    "30+ Days"
]

no_show_df["lead_time_group"] = pd.cut(
    no_show_df["booking_lead_days"],
    bins=bins,
    labels=labels
)

In [74]:
# Total appointments in each lead time group
lead_total = no_show_df["lead_time_group"].value_counts().sort_index()

# Only no-show appointments
lead_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["lead_time_group"]
    .value_counts()
    .sort_index()
)

# Combine both counts
lead_noshow = pd.concat([lead_total, lead_no_show], axis=1)

# Rename columns
lead_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
lead_noshow = lead_noshow.fillna(0)

# Convert to integer
lead_noshow["no_show_count"] = lead_noshow["no_show_count"].astype(int)

# Calculate percentage
lead_noshow["no_show_rate"] = (
    lead_noshow["no_show_count"] /
    lead_noshow["total_appointments"]
) * 100

# Convert index into column
lead_noshow = lead_noshow.reset_index()

lead_noshow

,lead_time_group,total_appointments,no_show_count,no_show_rate
0,0-1 Days,30633,4575,14.934874
1,2-3 Days,15022,2269,15.104513
2,4-7 Days,15897,2623,16.499969
3,8-14 Days,8758,1648,18.817082
4,15-30 Days,2342,469,20.025619
5,30+ Days,0,0,NaN


In [75]:
fig = px.line(
    lead_noshow,
    x="lead_time_group",
    y="no_show_rate",
    markers=True,
    title="No-Show Rate by Booking Lead Time"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Booking Lead Time",
    yaxis_title="No-Show Rate (%)"
)

fig.show()

In [76]:
### Business Insight

#- The chart compares no-show rates across different booking lead time groups.
#- If longer lead-time bookings show higher no-show rates, Apollo can send stronger reminder campaigns for those patients.
#- Patients booking close to the appointment date may have different attendance behavior than those booking weeks in advance.

In [77]:
### 3. No-Show Rate by Time Slot

#Analyzing whether Morning, Afternoon, and Evening appointments have different no-show rates.

In [78]:
# Total appointments in each time slot
slot_total = no_show_df["time_slot"].value_counts()

# Only no-show appointments
slot_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["time_slot"]
    .value_counts()
)

# Combine both counts
slot_noshow = pd.concat([slot_total, slot_no_show], axis=1)

# Rename columns
slot_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
slot_noshow = slot_noshow.fillna(0)

# Convert to integer
slot_noshow["no_show_count"] = slot_noshow["no_show_count"].astype(int)

# Calculate no-show rate
slot_noshow["no_show_rate"] = (
    slot_noshow["no_show_count"] /
    slot_noshow["total_appointments"]
) * 100

# Convert index into column
slot_noshow = slot_noshow.reset_index()

# Rename column
slot_noshow = slot_noshow.rename(columns={"index": "time_slot"})

slot_noshow

,time_slot,total_appointments,no_show_count,no_show_rate
0,10:45,2921,392,13.420062
1,10:00,2875,371,12.904348
2,10:30,2858,407,14.240728
3,10:15,2790,344,12.329749
4,11:45,2212,288,13.019892
5,11:15,2210,284,12.850679
6,09:30,2209,305,13.807153
7,09:45,2204,297,13.475499
8,09:15,2197,286,13.017751
9,11:00,2185,258,11.807780


In [79]:
fig = px.bar(
    slot_noshow,
    x="time_slot",
    y="no_show_rate",
    color="time_slot",
    text="no_show_rate",
    title="No-Show Rate by Time Slot"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Time Slot",
    yaxis_title="No-Show Rate (%)",
    showlegend=False
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [80]:
### No-Show Rate: Weekend vs Weekday

#Comparing patient attendance on weekends and weekdays.

In [82]:
# Create Weekend column

no_show_df["is_weekend"] = no_show_df["appointment_day"].isin(
    ["Saturday", "Sunday"]
)

no_show_df[["appointment_day", "is_weekend"]].head()

,appointment_day,is_weekend
0,Wednesday,False
1,Monday,False
2,Monday,False
3,Monday,False
4,Friday,False


In [83]:
# Total appointments
weekend_total = no_show_df["is_weekend"].value_counts()

# Only no-show appointments
weekend_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["is_weekend"]
    .value_counts()
)

# Combine both counts
weekend_noshow = pd.concat([weekend_total, weekend_no_show], axis=1)

# Rename columns
weekend_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
weekend_noshow = weekend_noshow.fillna(0)

# Convert to integer
weekend_noshow["no_show_count"] = weekend_noshow["no_show_count"].astype(int)

# Calculate no-show rate
weekend_noshow["no_show_rate"] = (
    weekend_noshow["no_show_count"] /
    weekend_noshow["total_appointments"]
) * 100

# Convert index into column
weekend_noshow = weekend_noshow.reset_index()

weekend_noshow

,is_weekend,total_appointments,no_show_count,no_show_rate
0,False,51731,7676,14.838298
1,True,20921,3908,18.679795


In [84]:
fig = px.bar(
    weekend_noshow,
    x="is_weekend",
    y="no_show_rate",
    color="is_weekend",
    text="no_show_rate",
    title="Weekend vs Weekday No-Show Rate"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Weekend",
    yaxis_title="No-Show Rate (%)",
    showlegend=False
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [85]:
### Business Insight

#- The time slot analysis compares attendance across Morning, Afternoon, and Evening appointments.
#- The weekend analysis compares attendance between weekdays and weekends.
#- Together with the previous booking lead-time analysis, these findings help Apollo identify when additional reminders or scheduling adjustments may reduce no-shows.

In [87]:
### 4. No-Show Rate by Appointment Type

#Analyzing which appointment types have the highest no-show risk.

In [88]:
# Total appointments by appointment type
type_total = no_show_df["appointment_type"].value_counts()

# Only no-show appointments
type_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["appointment_type"]
    .value_counts()
)

# Combine both counts
type_noshow = pd.concat([type_total, type_no_show], axis=1)

# Rename columns
type_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
type_noshow = type_noshow.fillna(0)

# Convert to integer
type_noshow["no_show_count"] = type_noshow["no_show_count"].astype(int)

# Calculate no-show rate
type_noshow["no_show_rate"] = (
    type_noshow["no_show_count"] /
    type_noshow["total_appointments"]
) * 100

# Convert index into column
type_noshow = type_noshow.reset_index()

# Rename column
type_noshow = type_noshow.rename(columns={"index": "appointment_type"})

type_noshow

,appointment_type,total_appointments,no_show_count,no_show_rate
0,In-Clinic,47561,5795,12.184353
1,Video Consult,15349,3232,21.056746
2,Home Visit,9742,2557,26.247177


In [89]:
fig = px.bar(
    type_noshow,
    x="appointment_type",
    y="no_show_rate",
    color="appointment_type",
    text="no_show_rate",
    title="No-Show Rate by Appointment Type"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Appointment Type",
    yaxis_title="No-Show Rate (%)",
    showlegend=False
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [90]:
### No-Show Rate by Booking Channel

#Comparing no-show risk across different booking channels.

In [91]:
# Total appointments by booking channel
channel_total = no_show_df["booking_channel"].value_counts()

# Only no-show appointments
channel_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["booking_channel"]
    .value_counts()
)

# Combine both counts
channel_noshow = pd.concat([channel_total, channel_no_show], axis=1)

# Rename columns
channel_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
channel_noshow = channel_noshow.fillna(0)

# Convert to integer
channel_noshow["no_show_count"] = channel_noshow["no_show_count"].astype(int)

# Calculate no-show rate
channel_noshow["no_show_rate"] = (
    channel_noshow["no_show_count"] /
    channel_noshow["total_appointments"]
) * 100

# Convert index into column
channel_noshow = channel_noshow.reset_index()

# Rename column
channel_noshow = channel_noshow.rename(columns={"index": "booking_channel"})

# Sort from highest to lowest
channel_noshow = channel_noshow.sort_values(
    "no_show_rate",
    ascending=False
)

channel_noshow

,booking_channel,total_appointments,no_show_count,no_show_rate
3,Walk-In,7281,1520,20.876253
4,Partner App,3594,651,18.113523
1,Website,17964,3090,17.201069
2,Call Centre,10855,1650,15.200368
0,Apollo App,32958,4673,14.178652


In [92]:
fig = px.bar(
    channel_noshow,
    x="no_show_rate",
    y="booking_channel",
    orientation="h",
    color="no_show_rate",
    text="no_show_rate",
    title="No-Show Rate by Booking Channel"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="No-Show Rate (%)",
    yaxis_title="Booking Channel"
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [93]:
### Business Insight

#- The appointment type analysis identifies which consultation mode has the highest no-show risk.
#- The booking channel analysis compares attendance across Apollo App, Website, Call Centre, Walk-in, and Partner channels.
#- These findings help Apollo target high-risk appointment types and booking channels with stronger reminder campaigns and scheduling improvements.

In [94]:
# Reminder and Engagement Effectiveness

### 1. No-Show Rate by Reminder Type

#Analyzing whether reminder methods such as SMS, WhatsApp, Call, and Email reduce no-show rates compared to patients who received no reminder.

In [95]:
# Total appointments by reminder type
reminder_total = no_show_df["reminder_type"].value_counts()

# Only no-show appointments
reminder_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["reminder_type"]
    .value_counts()
)

# Combine both counts
reminder_noshow = pd.concat([reminder_total, reminder_no_show], axis=1)

# Rename columns
reminder_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
reminder_noshow = reminder_noshow.fillna(0)

# Convert to integer
reminder_noshow["no_show_count"] = reminder_noshow["no_show_count"].astype(int)

# Calculate no-show rate
reminder_noshow["no_show_rate"] = (
    reminder_noshow["no_show_count"] /
    reminder_noshow["total_appointments"]
) * 100

# Convert index into column
reminder_noshow = reminder_noshow.reset_index()

# Rename column
reminder_noshow = reminder_noshow.rename(columns={"index": "reminder_type"})

# Highest to lowest no-show rate
reminder_noshow = reminder_noshow.sort_values(
    "no_show_rate",
    ascending=False
)

reminder_noshow

,reminder_type,total_appointments,no_show_count,no_show_rate
3,No Reminder,10977,3311,30.163068
2,SMS Only,14527,2363,16.266263
0,SMS + WhatsApp,25273,3414,13.508487
1,SMS + WhatsApp + Call,21875,2496,11.410286


In [96]:
fig = px.bar(
    reminder_noshow,
    x="no_show_rate",
    y="reminder_type",
    orientation="h",
    color="no_show_rate",
    text="no_show_rate",
    title="No-Show Rate by Reminder Type"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="No-Show Rate (%)",
    yaxis_title="Reminder Type"
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [97]:
### Business Insight

#- The chart compares no-show rates across different reminder methods.
#- Patients receiving reminders can be compared with those receiving no reminder.
#- The most effective reminder channel can help Apollo reduce missed appointments through targeted communication.

In [98]:
### 2. Prior No-Show History vs Future No-Show Rate

#Analyzing whether patients with a previous no-show history are more likely to miss future appointments.

In [102]:
# Create Yes/No column based on previous no-shows

no_show_df["prior_no_show_history"] = no_show_df["patient_prior_no_shows"].apply(
    lambda x: "Yes" if x > 0 else "No"
)

# Check the new column
no_show_df[["patient_prior_no_shows", "prior_no_show_history"]].head()

,patient_prior_no_shows,prior_no_show_history
0,0,No
1,0,No
2,0,No
3,0,No
4,0,No


In [103]:
# Total appointments
history_total = no_show_df["prior_no_show_history"].value_counts()

# Only no-show appointments
history_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["prior_no_show_history"]
    .value_counts()
)

# Combine both counts
history_noshow = pd.concat([history_total, history_no_show], axis=1)

# Rename columns
history_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
history_noshow = history_noshow.fillna(0)

# Convert to integer
history_noshow["no_show_count"] = history_noshow["no_show_count"].astype(int)

# Calculate no-show rate
history_noshow["no_show_rate"] = (
    history_noshow["no_show_count"] /
    history_noshow["total_appointments"]
) * 100

# Convert index into column
history_noshow = history_noshow.reset_index()

history_noshow

,prior_no_show_history,total_appointments,no_show_count,no_show_rate
0,No,63014,9549,15.153775
1,Yes,9638,2035,21.114339


In [104]:
fig = px.bar(
    history_noshow,
    x="prior_no_show_history",
    y="no_show_rate",
    color="prior_no_show_history",
    text="no_show_rate",
    title="No-Show Rate by Prior No-Show History"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Prior No-Show History",
    yaxis_title="No-Show Rate (%)",
    showlegend=False
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [105]:
### Business Insight

#- Patients with a previous no-show history can be compared with patients who have never missed an appointment before.
#- If the **Yes** group has a higher no-show rate, Apollo can identify these patients as high-risk and prioritize stronger reminder campaigns or follow-up calls.

In [106]:
### 3. Apollo Membership vs No-Show Rate

#Comparing no-show rates between Apollo members and non-members.

In [107]:
# Create a clean membership column

no_show_df["membership_status"] = no_show_df["membership_type"].fillna("Non-Member")

# Total appointments
member_total = no_show_df["membership_status"].value_counts()

# Only no-show appointments
member_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["membership_status"]
    .value_counts()
)

# Combine both counts
member_noshow = pd.concat([member_total, member_no_show], axis=1)

# Rename columns
member_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
member_noshow = member_noshow.fillna(0)

# Convert to integer
member_noshow["no_show_count"] = member_noshow["no_show_count"].astype(int)

# Calculate no-show rate
member_noshow["no_show_rate"] = (
    member_noshow["no_show_count"] /
    member_noshow["total_appointments"]
) * 100

# Convert index into column
member_noshow = member_noshow.reset_index()

member_noshow

,membership_status,total_appointments,no_show_count,no_show_rate
0,Non-Member,56428,9530,16.888779
1,Apollo Gold,8167,1033,12.648463
2,Apollo Silver,8057,1021,12.672211


In [109]:
fig = px.bar(
    member_noshow,
    x="membership_status",
    y="no_show_rate",
    color="membership_status",
    text="no_show_rate",
    title="No-Show Rate by Membership Status"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Membership Status",
    yaxis_title="No-Show Rate (%)",
    showlegend=False
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [110]:
### Repeat Patients vs First-Time Patients

#Comparing no-show rates between repeat patients and first-time patients.

In [115]:
# Total appointments
repeat_total = no_show_df["repeat_visit"].value_counts()

# Only no-show appointments
repeat_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["repeat_visit"]
    .value_counts()
)

# Combine both counts
repeat_noshow = pd.concat([repeat_total, repeat_no_show], axis=1)

# Rename columns
repeat_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
repeat_noshow = repeat_noshow.fillna(0)

# Convert to integer
repeat_noshow["no_show_count"] = repeat_noshow["no_show_count"].astype(int)

# Calculate no-show rate
repeat_noshow["no_show_rate"] = (
    repeat_noshow["no_show_count"] /
    repeat_noshow["total_appointments"]
) * 100

# Convert index into column
repeat_noshow = repeat_noshow.reset_index()

repeat_noshow

,repeat_visit,total_appointments,no_show_count,no_show_rate
0,Yes,63470,5452,8.589885
1,No,9182,6132,66.782836


In [116]:
fig = px.bar(
    repeat_noshow,
    x="repeat_visit",
    y="no_show_rate",
    color="repeat_visit",
    text="no_show_rate",
    title="No-Show Rate: Repeat vs First-Time Patients"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Repeat Visit",
    yaxis_title="No-Show Rate (%)",
    showlegend=False
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [117]:
### Business Insight

#- The chart compares attendance between repeat patients and first-time patients.
#- If repeat patients have a lower no-show rate, it suggests that familiarity with Apollo's services improves appointment attendance.
#- This insight can support patient retention and loyalty initiatives.

In [118]:
# Patient and Demographic Segmentation

### 1. No-Show Rate by Age Group

##Analyzing whether different age groups have different no-show rates.

In [119]:
# Total appointments by age group
age_total = no_show_df["age_group"].value_counts()

# Only no-show appointments
age_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["age_group"]
    .value_counts()
)

# Combine both counts
age_noshow = pd.concat([age_total, age_no_show], axis=1)

# Rename columns
age_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
age_noshow = age_noshow.fillna(0)

# Convert no_show_count to integer
age_noshow["no_show_count"] = age_noshow["no_show_count"].astype(int)

# Calculate no-show rate
age_noshow["no_show_rate"] = (
    age_noshow["no_show_count"] /
    age_noshow["total_appointments"]
) * 100

# Convert index into column
age_noshow = age_noshow.reset_index()

age_noshow

,age_group,total_appointments,no_show_count,no_show_rate
0,Adult (31-45),15133,2731,18.046653
1,Middle-aged (46-60),14862,2491,16.760867
2,Senior (60+),14516,2036,14.025902
3,Child (0-12),11745,1590,13.537676
4,Young Adult (19-30),10017,1797,17.939503
5,Teen (13-18),6379,939,14.720176


In [120]:
fig = px.bar(
    age_noshow,
    x="age_group",
    y="no_show_rate",
    color="age_group",
    text="no_show_rate",
    title="No-Show Rate by Age Group"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Age Group",
    yaxis_title="No-Show Rate (%)",
    showlegend=False
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [121]:
### Business Insight

#- The chart compares no-show rates across different age groups.
#- It helps identify which age groups are more likely to miss appointments.
#- These insights can help Apollo design age-specific reminder strategies and patient engagement programs.

In [122]:
### No-Show Rate by Gender

##Comparing attendance between male and female patients.

In [123]:
# Total appointments by gender
gender_total = no_show_df["patient_gender"].value_counts()

# Only no-show appointments
gender_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["patient_gender"]
    .value_counts()
)

# Combine both counts
gender_noshow = pd.concat([gender_total, gender_no_show], axis=1)

# Rename columns
gender_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
gender_noshow = gender_noshow.fillna(0)

# Convert to integer
gender_noshow["no_show_count"] = gender_noshow["no_show_count"].astype(int)

# Calculate no-show rate
gender_noshow["no_show_rate"] = (
    gender_noshow["no_show_count"] /
    gender_noshow["total_appointments"]
) * 100

# Convert index into column
gender_noshow = gender_noshow.reset_index()

gender_noshow

,patient_gender,total_appointments,no_show_count,no_show_rate
0,Female,37589,6029,16.039267
1,Male,35063,5555,15.842911


In [124]:
fig = px.bar(
    gender_noshow,
    x="patient_gender",
    y="no_show_rate",
    color="patient_gender",
    text="no_show_rate",
    title="No-Show Rate by Gender"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Gender",
    yaxis_title="No-Show Rate (%)",
    showlegend=False
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [125]:
### Business Insight

#- The chart compares no-show rates across different chronic condition categories.
#- Patients with ongoing health conditions may have different attendance behavior than patients without chronic conditions.
#- Apollo can use this insight to improve follow-up and reminder strategies for different patient groups.

In [126]:
### 2. Most Common Visit Reasons

#Analyzing which visit reasons contribute the highest number of appointments.

In [127]:
# Count appointments by visit reason

visit_reason_count = (
    df["visit_reason"]
    .value_counts()
    .reset_index()
)

# Rename columns
visit_reason_count.columns = ["visit_reason", "total_appointments"]

visit_reason_count.head(10)

,visit_reason,total_appointments
0,Follow-up,8807
1,Routine Checkup,8637
2,Chronic Condition Management,8380
3,Digestive Problem,8245
4,Fever & Infection,7041
5,Joint Pain,6664
6,Respiratory Issue,6590
7,Injury,6087
8,Eye Care,2959
9,Skin Issue,2862


In [128]:
fig = px.bar(
    visit_reason_count.head(10),
    x="visit_reason",
    y="total_appointments",
    color="visit_reason",
    text="total_appointments",
    title="Top 10 Most Common Visit Reasons"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Visit Reason",
    yaxis_title="Total Appointments",
    showlegend=False
)

fig.update_traces(textposition="outside")

fig.show()

In [129]:
### Visit Reasons with Higher No-Show Rate

#Analyzing which visit reasons are associated with higher patient dropout.

In [130]:
# Total appointments by visit reason
reason_total = no_show_df["visit_reason"].value_counts()

# Only no-show appointments
reason_no_show = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["visit_reason"]
    .value_counts()
)

# Combine both counts
reason_noshow = pd.concat([reason_total, reason_no_show], axis=1)

# Rename columns
reason_noshow.columns = ["total_appointments", "no_show_count"]

# Fill missing values
reason_noshow = reason_noshow.fillna(0)

# Convert to integer
reason_noshow["no_show_count"] = reason_noshow["no_show_count"].astype(int)

# Calculate no-show rate
reason_noshow["no_show_rate"] = (
    reason_noshow["no_show_count"] /
    reason_noshow["total_appointments"]
) * 100

# Convert index into column
reason_noshow = reason_noshow.reset_index()

# Rename column
reason_noshow = reason_noshow.rename(columns={"index": "visit_reason"})

# Sort from highest to lowest
reason_noshow = reason_noshow.sort_values(
    "no_show_rate",
    ascending=False
)

reason_noshow.head(10)

,visit_reason,total_appointments,no_show_count,no_show_rate
9,Skin Issue,2782,467,16.786485
5,Joint Pain,6469,1082,16.725924
11,Cardiac Concern,2639,430,16.294051
10,Mental Health,2733,445,16.282473
3,Digestive Problem,7984,1297,16.244990
7,Injury,5884,949,16.128484
4,Fever & Infection,6808,1095,16.084019
6,Respiratory Issue,6383,1020,15.979947
12,Women's Health,2362,377,15.961050
1,Routine Checkup,8377,1309,15.626119


In [131]:
fig = px.bar(
    reason_noshow.head(10),
    x="no_show_rate",
    y="visit_reason",
    orientation="h",
    color="no_show_rate",
    text="no_show_rate",
    title="Top 10 Visit Reasons with Highest No-Show Rate"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="No-Show Rate (%)",
    yaxis_title="Visit Reason"
)

fig.update_traces(texttemplate="%{text:.1f}%")

fig.show()

In [132]:
### Business Insight

#- The first chart identifies the most common reasons patients visit Apollo Hospitals.
#- The second chart highlights which visit reasons experience the highest no-show rates.
#- High-dropout visit reasons can be prioritized for stronger reminder campaigns and scheduling improvements.

In [133]:
### 3. Patient Age Distribution Across Selected Specialties

#Analyzing the age distribution of patients in Paediatrics, Gynaecology, and Psychiatry.

In [134]:
# Keep only the required specialties

selected_specialties = df[
    df["specialty_x"].isin([
        "Paediatrics",
        "Gynaecology",
        "Psychiatry"
    ])
]

selected_specialties.head()

,appointment_id,patient_id,doctor_id,appointment_date,appointment_day,appointment_day_of_week,appointment_month,appointment_quarter,appointment_year,appointment_hour,...,city_y,state_y,hospital_name_y,consultation_fee_y,rating,total_reviews,available_days,avg_slot_duration_min,accepts_insurance,teleconsult_enabled
1,APPT5000002,PAT128732,DOC1049,2023-03-13,Monday,0,3,Q1,2023,9,...,Pune,Maharashtra,Apollo Spectra Pune,1400,4.7,1912,Mon-Fri,15,No,Yes
6,APPT5000007,PAT123334,DOC1056,2023-07-28,Friday,4,7,Q3,2023,18,...,Kolkata,West Bengal,Apollo Clinic Park Street,3300,4.8,1225,Mon-Fri,15,No,No
14,APPT5000015,PAT117904,DOC1054,2023-11-06,Monday,0,11,Q4,2023,15,...,Delhi,Delhi,Apollo Spectra Delhi,1600,4.5,1562,Mon-Wed-Fri-Sat,10,Yes,No
19,APPT5000020,PAT109576,DOC1317,2023-09-14,Thursday,3,9,Q3,2023,15,...,Hyderabad,Telangana,Apollo Hospital Jubilee Hills,1400,4.9,623,Mon-Sat,10,Yes,No
24,APPT5000025,PAT128806,DOC1140,2024-03-24,Sunday,6,3,Q1,2024,10,...,Indore,Madhya Pradesh,Apollo Spectra Indore,2500,4.7,940,Mon-Sat,15,Yes,Yes


In [135]:
fig = px.histogram(
    selected_specialties,
    x="patient_age",
    color="specialty_x",
    nbins=25,
    barmode="overlay",
    title="Patient Age Distribution in Paediatrics, Gynaecology, and Psychiatry"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Patient Age",
    yaxis_title="Number of Patients"
)

fig.show()

In [136]:
### Business Insight

#- The histogram compares patient age distributions across Paediatrics, Gynaecology, and Psychiatry.
#- It helps identify which age groups are most common in each specialty.
#- These insights support age-specific service planning and resource allocation at Apollo Hospitals.

In [137]:
# Financial Performance

#|This section analyzes revenue, payment behavior, and insurance impact using only completed appointments, as revenue is zero for non-completed appointments.

In [138]:
# Keep only completed appointments for financial analysis

completed_df = df[df["appointment_status"] == "Completed"].copy()

print("Completed Appointments:", completed_df.shape)

Completed Appointments: (55275, 66)


In [139]:
### 1. Revenue Lost Due to No-Shows

#Estimating the consultation revenue lost because patients did not attend their scheduled appointments.

In [142]:
# Revenue lost from no-show appointments

# Estimated revenue lost due to no-show appointments

lost_revenue = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]["consultation_fee_x"]
    .sum()
)

print(f"Estimated Revenue Lost: ₹{lost_revenue:,.2f}")

Estimated Revenue Lost: ₹19,725,460.00


In [144]:
# Revenue loss by specialty

loss_specialty = (
    no_show_df[no_show_df["appointment_status"] == "No-Show"]
    .groupby("specialty_x")["consultation_fee_x"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

loss_specialty.columns = ["specialty", "revenue_lost"]

loss_specialty.head()

,specialty,revenue_lost
0,Dermatology,2603230
1,General Physician,2475640
2,Psychiatry,2336685
3,Orthopaedics,1586750
4,Cardiology,1569230


In [145]:
fig = px.bar(
    loss_specialty.head(10),
    x="revenue_lost",
    y="specialty",
    orientation="h",
    color="revenue_lost",
    text="revenue_lost",
    title="Top 10 Specialties by Revenue Lost Due to No-Shows"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Revenue Lost (₹)",
    yaxis_title="Specialty"
)

fig.show()

In [146]:
### Business Insight

#- The total estimated revenue lost highlights the financial impact of missed appointments.
#- The specialty-wise analysis identifies which departments lose the most potential consultation revenue.
#- Apollo can prioritize stronger reminder campaigns in these high-loss specialties.

In [147]:
### 2. Average Revenue by Specialty

#Comparing average revenue generated across medical specialties.

In [149]:
# Calculate average revenue by specialty

avg_specialty = (
    completed_df.groupby("specialty_x")["revenue_realized"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

# Rename columns
avg_specialty.columns = ["specialty", "average_revenue"]

avg_specialty.head(10)

,specialty,average_revenue
0,Psychiatry,2640.447906
1,Neurology,2076.479354
2,Urology,1988.621168
3,Cardiology,1940.765326
4,Gastroenterology,1896.815923
5,Endocrinology,1812.684840
6,Pulmonology,1803.229475
7,Orthopaedics,1547.197899
8,Gynaecology,1499.591895
9,Ophthalmology,1467.125160


In [150]:
fig = px.bar(
    avg_specialty.head(10),
    x="average_revenue",
    y="specialty",
    orientation="h",
    color="average_revenue",
    text="average_revenue",
    title="Top 10 Specialties by Average Revenue"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Average Revenue (₹)",
    yaxis_title="Specialty"
)

fig.update_traces(texttemplate="₹%{text:.0f}")

fig.show()

In [151]:
### Business Insight

#- The chart compares the average revenue earned from completed appointments across specialties.
#- Higher-revenue specialties contribute more value per consultation.
#- These insights help Apollo identify high-value departments for resource planning.

In [152]:
### Average Revenue by Appointment Type

#Comparing average revenue across Clinic, Video, and Home Visit appointments.

In [153]:
# Calculate average revenue by appointment type

avg_type = (
    completed_df.groupby("appointment_type")["revenue_realized"]
    .mean()
    .reset_index()
)

# Rename columns
avg_type.columns = ["appointment_type", "average_revenue"]

avg_type

,appointment_type,average_revenue
0,Home Visit,1960.048322
1,In-Clinic,1299.829959
2,Video Consult,1097.695441


In [154]:
fig = px.bar(
    avg_type,
    x="appointment_type",
    y="average_revenue",
    color="appointment_type",
    text="average_revenue",
    title="Average Revenue by Appointment Type"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Appointment Type",
    yaxis_title="Average Revenue (₹)",
    showlegend=False
)

fig.update_traces(texttemplate="₹%{text:.0f}")

fig.show()

In [155]:
### Business Insight

#- The chart compares average revenue across different appointment types.
#- It helps identify which consultation mode generates the highest revenue per completed appointment.
#- Apollo can use this insight to optimize its mix of clinic, video, and home visit services.

In [156]:
### 3. Revenue Generated by City

#Analyzing which cities contribute the highest revenue from completed appointments.

In [157]:
# Total revenue by city

city_revenue = (
    completed_df.groupby("city_x")["revenue_realized"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

# Rename columns
city_revenue.columns = ["city", "total_revenue"]

city_revenue.head(10)

,city,total_revenue
0,Bengaluru,9990616
1,Delhi,9958247
2,Mumbai,8989613
3,Hyderabad,7503245
4,Chennai,7188473
5,Kolkata,6543084
6,Pune,4734139
7,Ahmedabad,3956890
8,Lucknow,3177444
9,Indore,2223207


In [158]:
fig = px.bar(
    city_revenue.head(10),
    x="total_revenue",
    y="city",
    orientation="h",
    color="total_revenue",
    text="total_revenue",
    title="Top 10 Cities by Revenue"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Total Revenue (₹)",
    yaxis_title="City"
)

fig.update_traces(texttemplate="₹%{text:.0f}")

fig.show()

In [159]:
### Payment Mode Mix

#Analyzing the distribution of payment methods used for completed appointments.

In [160]:
# Count completed appointments by payment mode

payment_mix = (
    completed_df["payment_mode"]
    .value_counts()
    .reset_index()
)

# Rename columns
payment_mix.columns = ["payment_mode", "total_payments"]

payment_mix

,payment_mode,total_payments
0,Insurance,11641
1,Debit Card,7353
2,Net Banking,7349
3,Apollo Pay,7324
4,UPI,7259
5,Credit Card,7213
6,Cash,7136


In [161]:
fig = px.pie(
    payment_mix,
    names="payment_mode",
    values="total_payments",
    hole=0.4,
    title="Payment Mode Distribution"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5
)

fig.show()

In [162]:
### Business Insight

#- The revenue chart identifies which cities contribute the highest revenue from completed appointments.
#- The payment mode chart shows how patients prefer to pay for their consultations.
#- These insights help Apollo understand regional revenue performance and optimize payment services based on patient preferences.

In [163]:
### 4. Insurance Coverage vs Out-of-Pocket Payment

#Analyzing how insurance coverage affects the amount patients pay after completing their appointments.

In [164]:
# Keep "None" as a valid category (No Insurance)

insurance_payment = completed_df.copy()

insurance_payment["insurance_provider"] = (
    insurance_payment["insurance_provider"]
    .fillna("No Insurance")
)

# Calculate average amount paid by patients
insurance_summary = (
    insurance_payment.groupby("insurance_provider")["actual_fee_charged"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

# Rename columns
insurance_summary.columns = [
    "insurance_provider",
    "average_out_of_pocket"
]

insurance_summary

,insurance_provider,average_out_of_pocket
0,HDFC ERGO,1572.986274
1,No Insurance,1562.820670
2,Star Health,1561.374929
3,New India Assurance,1551.799889
4,Bajaj Allianz,1550.931503
5,Niva Bupa,1545.076250


In [165]:
fig = px.bar(
    insurance_summary,
    x="insurance_provider",
    y="average_out_of_pocket",
    color="insurance_provider",
    text="average_out_of_pocket",
    title="Average Out-of-Pocket Payment by Insurance Provider"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Insurance Provider",
    yaxis_title="Average Amount Paid (₹)",
    showlegend=False
)

fig.update_traces(texttemplate="₹%{text:.0f}")

fig.show()

In [166]:
### Business Insight

# The chart compares the average amount paid by patients across different insurance providers.
#- Patients without insurance can be directly compared with insured patients.
#- This analysis helps Apollo understand how insurance influences patient payments and supports pricing and insurance partnership decisions.

In [167]:
# Doctor Utilisation and Service Quality

### 1. Doctor Utilisation by Specialty

#Analyzing which specialties have the highest and lowest average doctor utilisation rates.

In [168]:
# Average doctor utilisation by specialty

utilization_specialty = (
    completed_df.groupby("specialty_x")["doctor_utilization_pct"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

# Rename columns
utilization_specialty.columns = [
    "specialty",
    "average_utilization"
]

utilization_specialty.head(10)

,specialty,average_utilization
0,Gastroenterology,77.256288
1,Endocrinology,77.146277
2,Cardiology,76.814200
3,Gynaecology,76.804531
4,Paediatrics,76.664143
5,Neurology,76.632342
6,General Physician,76.590615
7,Urology,76.442568
8,Orthopaedics,76.430485
9,Dermatology,76.420516


In [169]:
fig = px.bar(
    utilization_specialty,
    x="average_utilization",
    y="specialty",
    orientation="h",
    color="average_utilization",
    text="average_utilization",
    title="Average Doctor Utilisation by Specialty"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Average Utilisation (%)",
    yaxis_title="Specialty"
)

fig.update_traces(texttemplate="%{text:.1f}")

fig.show()

In [170]:
### Business Insight

#-The chart ranks specialties by average doctor utilisation.
#- High-utilisation specialties indicate heavier doctor workloads.
#- Lower-utilisation specialties may present opportunities for scheduling optimization and resource balancing.

In [171]:
### 2. Average Waiting Time by Specialty

#Analyzing whether patients wait longer in some specialties than others.

In [173]:
# Average waiting time by specialty

wait_specialty = (
    completed_df.groupby("specialty_x")["wait_time_minutes"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

# Rename columns
wait_specialty.columns = ["specialty", "average_wait_time"]

wait_specialty.head(10)

,specialty,average_wait_time
0,Ophthalmology,12.017454
1,Urology,11.905147
2,Dermatology,11.771648
3,Orthopaedics,11.701838
4,Neurology,11.638112
5,Gynaecology,11.616784
6,Cardiology,11.594240
7,General Physician,11.531807
8,Endocrinology,11.459441
9,Gastroenterology,11.390467


In [174]:
fig = px.bar(
    wait_specialty.head(10),
    x="average_wait_time",
    y="specialty",
    orientation="h",
    color="average_wait_time",
    text="average_wait_time",
    title="Top 10 Specialties by Average Waiting Time"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Average Waiting Time (Minutes)",
    yaxis_title="Specialty"
)

fig.update_traces(texttemplate="%{text:.1f}")

fig.show()

In [175]:
### Average Waiting Time by Time Slot

#Comparing patient waiting time across Morning, Afternoon, and Evening appointments.

In [176]:
# Average waiting time by time slot

wait_slot = (
    completed_df.groupby("time_slot")["wait_time_minutes"]
    .mean()
    .reset_index()
)

# Rename columns
wait_slot.columns = ["time_slot", "average_wait_time"]

wait_slot

,time_slot,average_wait_time
0,08:00,11.456382
1,08:15,11.446619
2,08:30,11.740260
3,08:45,11.656854
4,09:00,11.359712
5,09:15,11.519451
6,09:30,11.023296
7,09:45,11.441551
8,10:00,11.588624
9,10:15,11.430549


In [177]:
fig = px.bar(
    wait_slot,
    x="time_slot",
    y="average_wait_time",
    color="time_slot",
    text="average_wait_time",
    title="Average Waiting Time by Time Slot"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Time Slot",
    yaxis_title="Average Waiting Time (Minutes)",
    showlegend=False
)

fig.update_traces(texttemplate="%{text:.1f}")

fig.show()

In [178]:
### Business Insight

#- The specialty analysis identifies departments where patients experience longer waiting times.
#- The time slot analysis shows whether Morning, Afternoon, or Evening appointments experience longer delays.
#- These insights help Apollo improve scheduling efficiency and reduce patient waiting time.

In [179]:
### 3. Consultation Duration vs Patient Satisfaction

#Analyzing whether longer consultations are associated with higher patient satisfaction scores.

In [191]:
# Average satisfaction score for each consultation duration

duration_satisfaction = (
    satisfaction_df.groupby("consultation_duration_min")["patient_satisfaction_score"]
    .mean()
    .reset_index()
)

duration_satisfaction.head()

,consultation_duration_min,patient_satisfaction_score
0,5.0,4.071442
1,6.0,4.081779
2,7.0,4.058451
3,8.0,4.060190
4,9.0,4.081356


In [192]:
fig = px.line(
    duration_satisfaction,
    x="consultation_duration_min",
    y="patient_satisfaction_score",
    markers=True,
    title="Average Patient Satisfaction by Consultation Duration"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Consultation Duration (Minutes)",
    yaxis_title="Average Patient Satisfaction Score"
)

fig.show()

In [184]:
### Business Insight

#- The scatter plot shows how patient satisfaction changes with consultation duration.
#- The trendline helps identify whether longer consultations are associated with higher satisfaction.
#- This insight can help Apollo balance consultation quality with doctor efficiency.

In [185]:
### 4. Doctor Experience vs Consultation Fee

#Analyzing whether doctors with more years of experience charge higher consultation fees.

In [188]:
# Average consultation fee for each experience level

experience_fee = (
    completed_df.groupby("experience_years")["consultation_fee_x"]
    .mean()
    .reset_index()
)

experience_fee.head()

,experience_years,consultation_fee_x
0,2,1361.258322
1,3,1170.693642
2,4,1434.047997
3,5,1331.001712
4,6,1683.068047


In [189]:
fig = px.line(
    experience_fee,
    x="experience_years",
    y="consultation_fee_x",
    markers=True,
    title="Average Consultation Fee by Doctor Experience"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    xaxis_title="Doctor Experience (Years)",
    yaxis_title="Average Consultation Fee (₹)"
)

fig.show()

In [190]:
### Business Insight

#- The line chart shows how the average consultation fee changes with doctors' years of experience.
#- A rising trend indicates that doctors with more experience generally charge higher consultation fees.
#- Less experienced doctors tend to have lower consultation fees, making them suitable for cost-conscious patients.
#- Apollo can use this relationship to design balanced pricing strategies while maintaining accessibility across different experience levels.
#- Understanding this trend also helps in workforce planning by ensuring an appropriate mix of senior and junior doctors across hospitals.

In [ ]:
### Project Conclusion

#This project analyzed 75,000 Apollo Hospital appointments from 2022 to 2024 to understand patient booking behavior, no-show patterns, engagement, financial performance, and doctor service quality using Python, Pandas, and Plotly.

### Key Findings

#- Appointment trends showed how booking volume changed across months and quarters.
#- No-show rates varied across specialties, cities, booking channels, reminder types, and patient segments, highlighting areas where attendance can be improved.
#- Patients with prior no-show history and different engagement behaviors showed different attendance patterns, helping identify high-risk patients.
#- Financial analysis estimated revenue lost due to no-shows and identified high-revenue specialties, cities, and appointment types.
#- Doctor utilization, waiting time, consultation duration, and satisfaction analysis provided insights into service quality and operational efficiency.
#- The relationship between doctor experience and consultation fees helped explain pricing differences across experience levels.

### Business Recommendations

#- Strengthen reminder campaigns for high-risk patients, longer lead-time bookings, and high no-show specialties.
#- Improve scheduling in specialties and time slots with longer waiting times.
#- Focus retention efforts on first-time and non-member patients to improve attendance.
#- Prioritize operational improvements in departments with high revenue loss from no-shows.
#- Use doctor utilization and experience insights to balance workloads and optimize resource allocation.

#Overall, this project demonstrates how Python-based Exploratory Data Analysis (EDA) can convert raw healthcare appointment data into actionable business insights that support better patient engagement, higher operational efficiency, and improved revenue management.